In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

# Step 1: Sample data
pairs = [
    ("hello", "bonjour"),
    ("hi", "salut"),
    ("thanks", "merci"),
    ("yes", "oui"),
    ("no", "non")
]

# Step 2: Create vocab
input_texts = []
target_texts = []
input_chars = set()
target_chars = set()

for input_text, target_text in pairs:
    target_text = '\t' + target_text + '\n'  # start and end tokens
    input_texts.append(input_text)
    target_texts.append(target_text)
    input_chars.update(list(input_text))
    target_chars.update(list(target_text))

input_chars = sorted(list(input_chars))
target_chars = sorted(list(target_chars))

input_idx = {c: i for i, c in enumerate(input_chars)}
target_idx = {c: i for i, c in enumerate(target_chars)}
reverse_target_idx = {i: c for c, i in target_idx.items()}

max_input_len = max(len(txt) for txt in input_texts)
max_target_len = max(len(txt) for txt in target_texts)


In [2]:
# Step 3: Encode data
encoder_input = np.zeros((len(input_texts), max_input_len, len(input_chars)))
decoder_input = np.zeros((len(input_texts), max_target_len, len(target_chars)))
decoder_target = np.zeros((len(input_texts), max_target_len, len(target_chars)))
decoder_target

array([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.,

In [3]:
for i, (inp, tgt) in enumerate(zip(input_texts, target_texts)):
    for t, ch in enumerate(inp):
        encoder_input[i, t, input_idx[ch]] = 1.0
    for t, ch in enumerate(tgt):
        decoder_input[i, t, target_idx[ch]] = 1.0
        if t > 0:
            decoder_target[i, t - 1, target_idx[ch]] = 1.0

In [4]:
# Step 4: Build RNN Model
encoder_inputs = Input(shape=(None, len(input_chars)))
enc_lstm, state_h, state_c = LSTM(128, return_state=True)(encoder_inputs)
encoder_states = [state_h, state_c]

In [5]:
decoder_inputs = Input(shape=(None, len(target_chars)))
dec_lstm = LSTM(128, return_sequences=True, return_state=True)
dec_out, _, _ = dec_lstm(decoder_inputs, initial_state=encoder_states)
dec_dense = Dense(len(target_chars), activation="softmax")
decoder_outputs = dec_dense(dec_out)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="rmsprop", loss="categorical_crossentropy")
model.fit([encoder_input, decoder_input], decoder_target, batch_size=2, epochs=300, verbose=0)

# Step 5: Inference Model
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(128,))
decoder_state_input_c = Input(shape=(128,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
dec_outputs, h, c = dec_lstm(decoder_inputs, initial_state=decoder_states_inputs)
dec_states = [h, c]
dec_outputs = dec_dense(dec_outputs)
decoder_model = Model([decoder_inputs] + decoder_states_inputs, [dec_outputs] + dec_states)

# Step 6: Translate function
def translate(input_seq):
    seq = np.zeros((1, max_input_len, len(input_chars)))
    for t, ch in enumerate(input_seq):
        if ch in input_idx:
            seq[0, t, input_idx[ch]] = 1.0
    states = encoder_model.predict(seq)

    tgt_seq = np.zeros((1, 1, len(target_chars)))
    tgt_seq[0, 0, target_idx['\t']] = 1.0

    output = ''
    while True:
        output_tokens, h, c = decoder_model.predict([tgt_seq] + states)
        idx = np.argmax(output_tokens[0, -1, :])
        ch = reverse_target_idx[idx]
        if ch == '\n' or len(output) > max_target_len:
            break
        output += ch
        tgt_seq = np.zeros((1, 1, len(target_chars)))
        tgt_seq[0, 0, idx] = 1.0
        states = [h, c]
    return output

# Test
print("Translate 'hello':", translate("hello"))
print("Translate 'thanks':", translate("thanks"))
print("Translate 'no':", translate("no"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 284ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 299ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Translate 'hello': bonjour
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Translate 'thanks': merci
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Translate 'no': non


In [6]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# Step 1: Sample sentences and labels
texts = [
    "I love this movie",     # positive
    "This film is great",    # positive
    "I hate this movie",     # negative
    "This film is awful"     # negative
]
labels = [1, 1, 0, 0]  # 1 = positive, 0 = negative

# Step 2: Tokenize words
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
word_index = tokenizer.word_index

sequences

[[2, 6, 1, 3], [1, 4, 5, 7], [2, 8, 1, 3], [1, 4, 5, 9]]

In [7]:
# Step 3: Pad sequences (make all same length)
max_len = max(len(seq) for seq in sequences)
padded = pad_sequences(sequences, maxlen=max_len)

padded

array([[2, 6, 1, 3],
       [1, 4, 5, 7],
       [2, 8, 1, 3],
       [1, 4, 5, 9]], dtype=int32)

In [8]:
# Step 4: Build model
model = Sequential()
model.add(Embedding(input_dim=len(word_index) + 1, output_dim=8, input_length=max_len))
model.add(SimpleRNN(16))
model.add(Dense(1, activation='sigmoid'))

# Step 5: Compile and train
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(padded, np.array(labels), epochs=10, verbose=1)

# Step 6: Test prediction
test_text = ["I love this film great"]
test_seq = tokenizer.texts_to_sequences(test_text)
test_pad = pad_sequences(test_seq, maxlen=max_len)

pred = model.predict(test_pad)
print("Positive" if pred[0][0] > 0.5 else "Negative")

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5000 - loss: 0.6943
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.5000 - loss: 0.6921
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.5000 - loss: 0.6900
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.5000 - loss: 0.6878
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.5000 - loss: 0.6857
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.7500 - loss: 0.6835
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.7500 - loss: 0.6813
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.7500 - loss: 0.6790
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.7500 - loss: 0.6767
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.7500 - loss: 0.6744
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
Positive


In [9]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, LSTM, Embedding, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

it = ["hi", "how are you"]
tt = ["<start> hallo <end>", "<start> wie geht es dir <end>"]

it_tok = Tokenizer(filters="")
tt_tok = Tokenizer(filters="")

it_tok.fit_on_texts(it)
tt_tok.fit_on_texts(tt)

its = it_tok.texts_to_sequences(it)
tts = tt_tok.texts_to_sequences(tt)

mi = max(len(i) for i in its)
mt = max(len(j) for j in tts)

ip = pad_sequences(its, maxlen=mi, padding="post")
tp = pad_sequences(tts, maxlen=mt, padding="post")

tp_in = tp[:, :-1]
tp_out = tp[:, 1:]
tp_out = np.expand_dims(tp_out, -1)

ei = Input(shape=(None,))
ecd = Embedding(len(it_tok.word_index)+1, 64)(ei)

_, h, c = LSTM(64, return_state=True)(ecd)

di = Input(shape=(None,))
dcd = Embedding(len(tt_tok.word_index)+1, 64)(di)

dlstm = LSTM(64, return_sequences=True, return_state=True)
dout, _, _ = dlstm(dcd, initial_state=[h, c])

do = Dense(len(tt_tok.word_index)+1, activation='softmax')(dout)

model = Model([ei, di], do)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

model.fit([ip, tp_in], tp_out, epochs=200, verbose=0)

def predict_sentence(s):
    ts = it_tok.texts_to_sequences([s])
    ts = pad_sequences(ts, maxlen=mi, padding="post")

    start = tt_tok.word_index["<start>"] # Fixed: use "<start>" instead of "start"
    end = tt_tok.word_index["<end>"]     # Fixed: use "<end>" instead of "end"

    dec_in = np.array([[start]])

    output = []

    for _ in range(mt):
        pred = model.predict([ts, dec_in], verbose=0)

        word_id = np.argmax(pred[0, -1, :])

        if word_id == end:
            break

        word = tt_tok.index_word.get(word_id, "")
        output.append(word)

        dec_in = np.append(dec_in, [[word_id]], axis=1)

    return " ".join(output)

print(predict_sentence("hi"))
print(predict_sentence("how are you"))

hallo
wie geht es dir


In [10]:

a=np.array(["hi","how"])

In [11]:
np.argmax(a)

np.int64(1)